# Mesh 5 · A canopy: panels, bars and a schedule

**Application, the gentlest.** Companion to [Week 05 §7](../README.md#7-attributes-data-living-on-geometry).
Uses notebooks 01 and 03.

![One warped quad with its two diagonals missing each other; the canopy coloured by warp; the canopy coloured by slope](../figures/app-canopy.svg)

The first architectural use of a mesh needs no algorithm at all: **a face is
a panel, an edge is a bar, a vertex is a node**. Once the canopy is a mesh,
the quantities a fabricator asks for are loops over it: how much glass, how
many different bars, which panels are not flat, where the rain goes.

Simple on purpose: nothing here goes beyond the measuring and attributes of
[`03_data_on_a_mesh`](03_data_on_a_mesh.ipynb).

In [ ]:
# Setup: run this cell first. It finds the shared drawing helper (viz.py),
# whether this notebook sits in Lecture/notebooks/ or is your copy in MyWork/.
import sys
import pathlib

HERE = pathlib.Path.cwd()
NB = next(p / "Lecture" / "notebooks" for p in [HERE, *HERE.parents]
          if (p / "Lecture" / "notebooks" / "viz.py").exists())
sys.path.insert(0, str(NB))
from viz import chart, draw_graph, draw_mesh, draw_points, ramp, row, view3d

import math
from collections import Counter

from compas.datastructures import Mesh

## Step 1 — the canopy
12 m by 8 m, 8 by 6 panels. The shape is one line: an arch along x that is
taller on one side than on the other. Change it later and rerun everything
below.

In [ ]:
canopy = Mesh.from_meshgrid(dx=12, nx=8, dy=8, ny=6)


def height(x, y):
    return 3.0 + (1.2 + 0.08 * (y - 4)) * math.sin(math.pi * x / 12)


for v in canopy.vertices():
    x, y, _ = canopy.vertex_coordinates(v)
    canopy.vertex_attribute(v, "z", height(x, y))

print("nodes", canopy.number_of_vertices(), "| bars", canopy.number_of_edges(),
      "| panels", canopy.number_of_faces())
draw_mesh(canopy, "the canopy", view="iso", vertex_labels=False)

In [ ]:
view3d((canopy, {"show_edges": True}))

## Step 2 — how much glass?
The plan is 12 × 8 = 96 m². The panels are tilted, so the glass is more than
that. Each face knows its own area.

In [ ]:
areas = {f: canopy.face_area(f) for f in canopy.faces()}
print(f"plan area   {12 * 8:.1f} m2")
print(f"glass area  {sum(areas.values()):.1f} m2   (mesh.area() agrees: {canopy.area():.1f})")
print(f"smallest panel {min(areas.values()):.2f} m2, largest {max(areas.values()):.2f} m2")

## Step 3 — how many different bars?
On a flat grid there are two bar lengths. On a shaped canopy nearly every bar
is different, and every different bar is a separate line in the cutting
list. Rounding to a tolerance groups them into **families**: the coarser the
tolerance, the fewer families, the cheaper the job.

In [ ]:
lengths = {e: canopy.edge_length(e) for e in canopy.edges()}
print(f"total bar length {sum(lengths.values()):.1f} m")
for tolerance in (0.001, 0.01, 0.05):
    families = Counter(round(length / tolerance) * tolerance for length in lengths.values())
    print(f"cut to the nearest {tolerance * 1000:>3.0f} mm -> {len(families):3d} different bars")

families = Counter(round(length / 0.05) * 0.05 for length in lengths.values())
print("\ncutting list at 50 mm:")
for length, count in sorted(families.items()):
    print(f"  {length:.2f} m  x {count}")

## Step 4 — is each panel flat?
Four points are not always on one plane. A quad panel is flat only if its
two diagonals cross; when they pass each other, the gap between them is the
**warp**, and flat glass cannot follow it. Measure the gap for one panel by
hand, then for all of them.

In [ ]:
from compas.geometry import Line, distance_line_line


def warp(mesh, face):
    a, b, c, d = [mesh.vertex_coordinates(v) for v in mesh.face_vertices(face)]
    return distance_line_line(Line(a, c), Line(b, d))


print(f"panel 0: the diagonals miss each other by {warp(canopy, 0) * 1000:.0f} mm")

warps = {f: warp(canopy, f) for f in canopy.faces()}
limit = 0.015        # what this (imaginary) glazier accepts, in metres
too_warped = [f for f in canopy.faces() if warps[f] > limit]
print(f"warp from {min(warps.values()) * 1000:.0f} to {max(warps.values()) * 1000:.0f} mm; "
      f"{len(too_warped)} of {len(warps)} panels exceed {limit * 1000:.0f} mm")

top = max(warps.values())
draw_mesh(canopy, "warp per panel (orange = worst)", vertex_labels=False,
          face_colors={f: ramp(w / top) for f, w in warps.items()})

The warp is largest at the two ends and smallest beside the crown. It comes from
one term only, `0.08 * (y - 4)`: the arch being taller on one side. An arch
of constant height curves in one direction and every panel on it is flat.
**Which term of your shape function costs money is something the mesh can
tell you.**

## Step 5 — where does the rain go?
A face normal gives the slope of a panel: the angle between the normal and
straight up. Nearly level panels pond.

In [ ]:
slope = {}
for f in canopy.faces():
    nz = canopy.face_normal(f)[2]
    slope[f] = math.degrees(math.acos(min(1.0, abs(nz))))

ponding = [f for f in canopy.faces() if slope[f] < 6.0]
print(f"slope from {min(slope.values()):.1f} to {max(slope.values()):.1f} degrees")
print(f"{len(ponding)} panels are flatter than 6 degrees:", ponding)
draw_mesh(canopy, "panels flatter than 6 degrees", vertex_labels=False, face_labels=True,
          highlight_faces=ponding)

## Step 6 — write the answers onto the mesh
Until now the results lived in loose dictionaries. Store them as **face
attributes** and the mesh becomes the schedule: one object to save, send and
query.

In [ ]:
for f in canopy.faces():
    canopy.face_attributes(f, ["area", "warp", "slope"], [areas[f], warps[f], slope[f]])
    canopy.face_attribute(f, "glass", "laminated" if slope[f] < 6.0 else "standard")

print("panel 20:", canopy.face_attributes(20))
print("laminated panels:", sorted(canopy.faces_where(glass="laminated")))
order = sum(canopy.face_attribute(f, "area") for f in canopy.faces_where(glass="standard"))
print(f"standard glass to order: {order:.1f} m2")

## Your turn
1. Set `0.08` to `0.0` in `height()` and rerun. What happens to the warp,
   and to the number of bar families at 1 mm?
2. Steepen the arch from `1.2` to `2.5`. Which numbers move: glass area,
   ponding panels, families?
3. Count the nodes by how many bars meet there (`vertex_degree`). How many
   different node types does the canopy need?
4. `quads_to_triangles()` on a copy makes every panel flat by construction.
   What does it do to the panel count and the total bar length?